# 💨 01d — Pipeline qualité de l'air (LCSQA / AASQA)

Construit `dim_qualite_air.parquet` (dept × annee_mois). Lancer
`00_config_commun.ipynb` avant.

Source : LCSQA, agrège les mesures des réseaux régionaux AASQA.
https://www.data.gouv.fr/datasets/donnees-temps-reel-de-mesure-des-concentrations-de-polluants-atmospheriques-reglementes-1

Deux fichiers sur la même page :

| Fichier | | Période |
|---|---|---|
| `AASQA/FR_E2_YYYY-MM-01.csv` | concentrations horaires par station et polluant | 2021-2025 (rien avant 2021) |
| `AASQA/stations_metadata.xls` | lat/lon de chaque station, feuille `AirQualityStations` | photo actuelle du réseau |

O₃ et PM2.5 déclenchent les crises d'asthme, NO₂ potentialise les réponses
allergiques, les particules fragilisent les voies respiratoires des
nourrissons (bronchiolite).

Colonnes produites :

| dim_qualite_air | Polluant | Description des variables |
|---|---|---|
| `no_moy` | NO | concentration mensuelle moyenne (µg/m³) |
| `no2_moy` | NO2 | facteur allergie + asthme |
| `o3_moy` | O3 | déclencheur crises d'asthme |
| `pm10_moy` | PM10 (< 10 µm) | facteur asthme & bronchiolite |
| `pm25_moy` | PM2.5 (< 2,5 µm) | facteur asthme sévère |
| `so2_moy` | SO2 | peu de stations, cf. limites plus bas |

On garde seulement les mesures `validité == 1`, et les valeurs <> 1 passent en NaN avant agrégation.

Pour attribuer chaque mesure au bon département, nous utilisons le fichier `stations_metadata.xls` comme référentiel.

Quelques remarques : 
 - 2020 absent (le flux temps réel LCSQAn'est archivé qu'à partir de 2021). 
 - SO₂ : peu de stations le mesurent.
 - 4 départements toujours sans données (09, 11, 46, 48). Pour 11/46/48 aucune station n'est recensée dans
`stations_metadata.xls` ; pour 09 il y a une station mais aucune mesure
exploitable. Ce sont des départements ruraux du sud-ouest peu peuplés.

In [36]:
"TODO : Dans la base, il y a la variable \"Type d'implantation\" qui indique si la station est urbaine, périurbaine ou rurale. On pourrait l'utiliser pour filtrer les stations urbaines (ou périurbaines) et ne garder que celles-ci pour le calcul des indicateurs de qualité de l'air ?"

'TODO : Dans la base, il y a la variable "Type d\'implantation" qui indique si la station est urbaine, périurbaine ou rurale. On pourrait l\'utiliser pour filtrer les stations urbaines (ou périurbaines) et ne garder que celles-ci pour le calcul des indicateurs de qualité de l\'air ?'

In [37]:
# Préambule : on se place dans le répertoire racine du projet et on ajoute le répertoire courant au PYTHONPATH pour pouvoir importer src/config.py

# pour recharger automatiquement les modules modifiés （src config surtout） sans redémarrer le kernel
%load_ext autoreload 
%autoreload 2

import os
import sys
from pathlib import Path
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
from src.config import RAW_DIR, TABLES_DIR, ANNEE_DEBUT, ANNEE_FIN, DEPTS,  DEPT_NOM_TO_CODE
from src.validation import valider_dim_table

print(
    f"Config chargée depuis src/config.py : {len(DEPTS)} départements | {ANNEE_DEBUT}–{ANNEE_FIN}")
print(f"RAW_DIR    = {RAW_DIR}")
print(f"TABLES_DIR = {TABLES_DIR}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Config chargée depuis src/config.py : 96 départements | 2020–2025
RAW_DIR    = /Users/siranh/Documents/Data Scientest/projet_liora/data/raw
TABLES_DIR = /Users/siranh/Documents/Data Scientest/projet_liora/data/processed


In [38]:
df = pd.read_csv(
    "data/raw/AASQA/FR_E2_2021-01-01.csv",
    sep=";", nrows=5
)
print("=== FR_E2 (mesures) ===")
print(df.columns.tolist())
display(df.head())

# Métadonnées des stations (pour la correspondance station -> département)
df_stations_check = pd.read_excel(
    "data/raw/AASQA/stations_metadata.xls",
    sheet_name="AirQualityStations", nrows=5
)
print("\n=== stations_metadata (AirQualityStations) ===")
print(df_stations_check.columns.tolist())
display(df_stations_check[["NatlStationCode", "Name", "Municipality", "Latitude", "Longitude"]])

=== FR_E2 (mesures) ===
['Date de début', 'Date de fin', 'Organisme', 'code zas', 'Zas', 'code site', 'nom site', "type d'implantation", 'Polluant', "type d'influence", 'discriminant', 'Réglementaire', "type d'évaluation", 'procédure de mesure', 'type de valeur', 'valeur', 'valeur brute', 'unité de mesure', 'taux de saisie', 'couverture temporelle', 'couverture de données', 'code qualité', 'validité']


,Date de début,Date de fin,Organisme,code zas,Zas,code site,nom site,type d'implantation,Polluant,type d'influence,...,procédure de mesure,type de valeur,valeur,valeur brute,unité de mesure,taux de saisie,couverture temporelle,couverture de données,code qualité,validité
0,2021/01/01 00:00:00,2021/01/01 01:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01005,Hayange,Périurbaine,PM10,Industrielle,...,Auto PM_Conf_app MP101M-RST,moyenne horaire validée,18.9,18.875,µg-m3,NaN,NaN,NaN,A,1
1,2021/01/01 01:00:00,2021/01/01 02:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01005,Hayange,Périurbaine,PM10,Industrielle,...,Auto PM_Conf_app MP101M-RST,moyenne horaire validée,10.8,10.800,µg-m3,NaN,NaN,NaN,A,1
2,2021/01/01 02:00:00,2021/01/01 03:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01005,Hayange,Périurbaine,PM10,Industrielle,...,Auto PM_Conf_app MP101M-RST,moyenne horaire validée,10.3,10.300,µg-m3,NaN,NaN,NaN,A,1
3,2021/01/01 03:00:00,2021/01/01 04:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01005,Hayange,Périurbaine,PM10,Industrielle,...,Auto PM_Conf_app MP101M-RST,moyenne horaire validée,6.4,6.400,µg-m3,NaN,NaN,NaN,A,1
4,2021/01/01 04:00:00,2021/01/01 05:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01005,Hayange,Périurbaine,PM10,Industrielle,...,Auto PM_Conf_app MP101M-RST,moyenne horaire validée,8.1,8.050,µg-m3,NaN,NaN,NaN,A,1



=== stations_metadata (AirQualityStations) ===
['GMLID', 'LocalId', 'Namespace', 'Version', 'NatlStationCode', 'Name', 'Municipality', 'EUStationCode', 'ActivityBegin', 'ActivityEnd', 'Latitude', 'Longitude', 'SRSName', 'Altitude', 'AltitudeUnit', 'AreaClassification', 'BelongsTo']


,NatlStationCode,Name,Municipality,Latitude,Longitude
0,FR19012,Brest Mace,BREST,48.386180,-4.486600
1,FR34051,Chateauroux Sud,CHÂTEAUROUX,46.798280,1.693139
2,FR12047,Bessières-ECONOTRE,BESSIÈRES,43.803140,1.595475
3,FR16034,Strasbourg Clemenceau,STRASBOURG,48.590430,7.744983
4,FR30026,Luneville,LUNÉVILLE,48.584824,6.483900


In [39]:
# Test reverse_geocoder

import reverse_geocoder as rg

# Lat/lon (stations_metadata.xls）
coords = [(48.386180, -4.486600), (48.8566, 2.3522)]
resultats = rg.search(coords)

for r in resultats:
    print(r)

# {'lat': '48.4', 'lon': '-4.48333', 'name': 'Brest',
#  'admin1': 'Brittany',                  (région): Bretagne
#  'admin2': 'Departement du Finistere',       (département): Finistère
#  'cc': 'FR'}
#
# {'lat': '48.85341', 'lon': '2.3488', 'name': 'Paris',
#  'admin1': 'Ile-de-France',                   (région): Île-de-France
#  'admin2': 'Paris',                           (département): Paris
#  'cc': 'FR'}

{'lat': '48.4', 'lon': '-4.48333', 'name': 'Brest', 'admin1': 'Brittany', 'admin2': 'Departement du Finistere', 'cc': 'FR'}
{'lat': '48.85341', 'lon': '2.3488', 'name': 'Paris', 'admin1': 'Ile-de-France', 'admin2': 'Paris', 'cc': 'FR'}


In [40]:
import re
import reverse_geocoder as rg

def build_station_dict_aasqa() -> dict:
    """
    Construit le dictionnaire {NatlStationCode vs. code_dept} à partir des
    métadonnées officielles des stations AASQA (feuille "AirQualityStations"
    de stations_metadata.xls) et d'un reverse geocoding (lat/lon → département).

    Sauvegarde aussi data/processed/stations_aasqa_dept.parquet pour réutilisation.

    Retourne : dict { NatlStationCode (str) → code_dept (str) }
    """
    stations_path = RAW_DIR / "AASQA" / "stations_metadata.xls"
    if not stations_path.exists():
        print(f"⚠️  Métadonnées stations introuvables : {stations_path}")
        return {}

    df_stations = pd.read_excel(stations_path, sheet_name="AirQualityStations")
    print(f"Stations totales : {len(df_stations)}")

    # Reverse geocoding : lat/lon → département
    coords = list(zip(df_stations["Latitude"], df_stations["Longitude"]))
    resultats = rg.search(coords)
    df_stations["dept_nom"] = [r["admin2"] for r in resultats]
    df_stations["pays"]     = [r["cc"]     for r in resultats]

    # Garder uniquement France métropolitaine
    df_fr = df_stations[df_stations["pays"] == "FR"].copy()
    print(f"Stations France  : {len(df_fr)}")

    # Nom département (reverse geocoder) → code INSEE (cf. src/config.py)
    df_fr["dept"] = df_fr["dept_nom"].map(DEPT_NOM_TO_CODE)

    # Diagnostics
    sans_code = df_fr[df_fr["dept"].isna()][["NatlStationCode", "Name", "dept_nom"]]
    if len(sans_code):
        print(f"⚠️  {len(sans_code)} stations sans correspondance département :")
        print(sans_code.to_string()) # checker si le nom du département est correct ou s'il faut l'ajouter à DEPT_NOM_TO_CODE (src.config.py)
    else:
        print("✅ Toutes les stations mappées")

    df_fr = df_fr.dropna(subset=["dept"])

    # Sauvegarder pour les runs futurs
    df_fr[["NatlStationCode", "Name", "dept"]].to_parquet(
        TABLES_DIR / "stations_aasqa_dept.parquet", index=False
    )
    print(f"\n Sauvegardé → data/processed/stations_aasqa_dept.parquet ({len(df_fr)} stations)")
    print(f"✅ Départements couverts : {df_fr['dept'].nunique()} / {len(DEPTS)}")

    return dict(zip(df_fr["NatlStationCode"], df_fr["dept"]))

In [41]:

def build_dim_qualite_air(station_to_dept: dict) -> pd.DataFrame:
    """
    Construit la table dim_qualite_air à partir des fichiers AASQA journaliers
    (data/raw/AASQA), le flux LCSQA est archivé jour par jour.

    SOURCE : LCSQA — Concentrations de polluants atmosphériques réglementés

    VARIABLES PRODUITES :
    ────────────────────
    dept          → code département
    annee_mois    → période (ex: "2021-03")
    pm10_moy      → concentration mensuelle moyenne PM10 (µg/m³)
    pm25_moy      → concentration mensuelle moyenne PM2.5
    no2_moy       → concentration mensuelle moyenne NO2
    o3_moy        → concentration mensuelle moyenne O3

    CLÉ PRIMAIRE : dept × annee_mois
    """

    aasqa_dir = RAW_DIR / "AASQA"
    if not aasqa_dir.exists():
        print(f"⚠️  Dossier manquant : {aasqa_dir}")
        return pd.DataFrame()

    fichiers = sorted(aasqa_dir.glob("FR_E2_*.csv"))

    print(f"Chargement de {len(fichiers)} fichiers AASQA journaliers.")

    # Polluants d'intérêt et leurs noms normalisés
    POLLUANTS = {
        "PM10":  "pm10",
        "PM2.5": "pm25",
        "NO2":   "no2",
        "O3":    "o3",
        "NO":    "no",    # optionnel
        "SO2":   "so2",   # optionnel
    }

    dfs = []

    for fpath in fichiers:

        # Extraire l'année et le mois depuis le nom de fichier
        # Format : FR_E2_2021-01-15.csv → annee_mois = "2021-01"
        # (plusieurs fichiers journaliers partagent le même annee_mois ; le
        # groupby plus bas les agrège ensemble en une vraie moyenne mensuelle)
        match = re.search(r'FR_E2_(\d{4}-\d{2})-\d{2}\.csv', fpath.name)
        if not match:
            continue
        annee_mois = match.group(1)

        try:
            df = pd.read_csv(
                fpath,
                sep=";",
                encoding="utf-8",
                low_memory=False,
                dtype=str          # tout en texte d'abord
            )

            # ── Colonnes utiles ──────────────────────────────────────────────
            # Renommer pour standardiser
            rename = {
                "Date de début":    "date_debut",
                "code site":        "code_site",
                "Polluant":         "polluant",
                "valeur":           "valeur",
                "validité":         "validite",
                "unité de mesure":  "unite",
            }
            df = df.rename(columns={k: v for k, v in rename.items()
                                    if k in df.columns})

            df["dept"] = df["code_site"].map(station_to_dept)
            df = df.dropna(subset=["dept"])
            df = df[df["dept"].isin(DEPTS)]

            # ── Filtrage sur les polluants d'intérêt ─────────────────────
            df = df[df["polluant"].isin(POLLUANTS.keys())].copy()
            if df.empty:
                continue

            # ── Filtrage sur données valides (validite = 1) ───────────────
            df["validite"] = pd.to_numeric(df["validite"], errors="coerce")
            df = df[df["validite"] == 1]

            # ── Conversion numérique de la valeur ─────────────────────────
            df["valeur"] = pd.to_numeric(df["valeur"], errors="coerce")
            df = df.dropna(subset=["valeur"])

            # ── Nettoyage des valeurs aberrantes ──────────────────────────
            # Valeurs négatives → NaN
            df.loc[df["valeur"] < 0, "valeur"] = np.nan

            # TODO : traiter les valeurs extrêmes ??

            df["annee_mois"] = annee_mois

            # ── Normaliser le nom du polluant ──────────────────────────────
            df["polluant_norm"] = df["polluant"].map(POLLUANTS)

            dfs.append(df[["dept", "annee_mois", "polluant_norm",
                           "valeur"]].copy())

        except Exception as e:
            print(f"  ⚠️  Erreur {fpath.name} : {e}")

    if not dfs:
        print("❌ Aucune donnée chargée")
        return pd.DataFrame()

    df_all = pd.concat(dfs, ignore_index=True)
    print(f"Total lignes valides : {len(df_all):,}")
    print(
        f"Polluants disponibles : {df_all['polluant_norm'].unique().tolist()}")

    # ── Agrégation mensuelle par département × polluant ───────────────────
    # Plusieurs fichiers journaliers par mois → groupby dept × mois × polluant
    df_pivot = (
        df_all.groupby(["dept", "annee_mois", "polluant_norm"])["valeur"]
        .mean()  # TODO : Mean ou max ?
        .round(2)
        .reset_index()
    )

    # Pivot : une colonne par polluant
    df_wide = df_pivot.pivot_table(
        index=["dept", "annee_mois"],
        columns="polluant_norm",
        values="valeur",
        aggfunc="mean"
    ).reset_index()

    # Renommer les colonnes
    df_wide.columns.name = None
    rename_cols = {p: f"{p}_moy" for p in POLLUANTS.values()
                   if p in df_wide.columns}
    df_wide = df_wide.rename(columns=rename_cols)

    print(f"\n✅ dim_qualite_air : {df_wide.shape[0]:,} lignes × "
          f"{df_wide.shape[1]} colonnes")
    print(f"Période      : {df_wide['annee_mois'].min()} → "
          f"{df_wide['annee_mois'].max()}")
    print(f"Départements : {df_wide['dept'].nunique()} couverts")
    print(f"Colonnes     : {list(df_wide.columns)}")

    # Couverture par polluant
    print("\n📊 Couverture par polluant :")
    for col in [c for c in df_wide.columns if c.endswith("_moy")]:
        pct = df_wide[col].notna().mean() * 100
        barre = "█" * int(pct // 10) + "░" * (10 - int(pct // 10))
        print(f"  {col:<20} {barre} {pct:.0f}%")

    return df_wide

In [42]:
# ══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════════════
print("Construction du dictionnaire station -> dept (AASQA)...")
station_to_dept_aasqa = build_station_dict_aasqa()

print("\nConstruction de dim_qualite_air...")
dim_qualite_air = build_dim_qualite_air(station_to_dept_aasqa)

if not dim_qualite_air.empty:
    valider_dim_table(dim_qualite_air, "dim_qualite_air")
    dim_qualite_air.to_parquet(TABLES_DIR / "dim_qualite_air.parquet", index=False)
    print(f"\nSauvegardé → data/processed/dim_qualite_air.parquet")
    display(dim_qualite_air.head(10))

Construction du dictionnaire station -> dept (AASQA)...
Stations totales : 868
Stations France  : 810
✅ Toutes les stations mappées

 Sauvegardé → data/processed/stations_aasqa_dept.parquet (810 stations)
✅ Départements couverts : 93 / 96

Construction de dim_qualite_air...
Chargement de 1826 fichiers AASQA journaliers.
Total lignes valides : 65,799,902
Polluants disponibles : ['pm10', 'no', 'no2', 'o3', 'pm25', 'so2']

✅ dim_qualite_air : 5,520 lignes × 8 colonnes
Période      : 2021-01 → 2025-12
Départements : 92 couverts
Colonnes     : ['dept', 'annee_mois', 'no_moy', 'no2_moy', 'o3_moy', 'pm10_moy', 'pm25_moy', 'so2_moy']

📊 Couverture par polluant :
  no_moy               █████████░ 98%
  no2_moy              █████████░ 98%
  o3_moy               ████████░░ 87%
  pm10_moy             █████████░ 98%
  pm25_moy             █████████░ 92%
  so2_moy              ██░░░░░░░░ 29%
── Validation de dim_qualite_air ──
  ✅ Tous les codes dept sont valides (92 départements)
  ✅ Aucun doublon 

,dept,annee_mois,no_moy,no2_moy,o3_moy,pm10_moy,pm25_moy,so2_moy
0,01,2021-01,4.86,16.41,36.29,12.46,10.70,1.40
1,01,2021-02,4.81,14.39,45.65,21.63,12.85,0.98
2,01,2021-03,3.53,12.62,56.51,15.84,11.10,1.48
3,01,2021-04,1.34,6.84,74.79,12.92,8.30,0.78
4,01,2021-05,1.25,6.32,66.95,7.44,4.59,0.66
5,01,2021-06,0.91,6.51,72.05,13.48,8.08,1.28
6,01,2021-07,1.28,6.88,60.52,12.63,8.15,1.85
7,01,2021-08,1.27,5.98,55.78,9.93,5.99,1.32
8,01,2021-09,2.51,10.78,56.53,11.12,6.80,0.71
9,01,2021-10,5.56,11.99,44.56,11.55,8.39,0.47


In [43]:
# ══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════════════
print("Construction du dictionnaire station -> dept (AASQA)...")
station_to_dept_aasqa = build_station_dict_aasqa()

print("\nConstruction de dim_qualite_air...")
dim_qualite_air = build_dim_qualite_air(station_to_dept_aasqa)

if not dim_qualite_air.empty:
    valider_dim_table(dim_qualite_air, "dim_qualite_air")
    dim_qualite_air.to_parquet(
        TABLES_DIR / "dim_qualite_air.parquet", index=False
    )
    print(f"\n✅ Sauvegardé → data/processed/dim_qualite_air.parquet")
    display(dim_qualite_air.head(10))

Construction du dictionnaire station -> dept (AASQA)...
Stations totales : 868
Stations France  : 810
✅ Toutes les stations mappées

 Sauvegardé → data/processed/stations_aasqa_dept.parquet (810 stations)
✅ Départements couverts : 93 / 96

Construction de dim_qualite_air...
Chargement de 1826 fichiers AASQA journaliers.
Total lignes valides : 65,799,902
Polluants disponibles : ['pm10', 'no', 'no2', 'o3', 'pm25', 'so2']

✅ dim_qualite_air : 5,520 lignes × 8 colonnes
Période      : 2021-01 → 2025-12
Départements : 92 couverts
Colonnes     : ['dept', 'annee_mois', 'no_moy', 'no2_moy', 'o3_moy', 'pm10_moy', 'pm25_moy', 'so2_moy']

📊 Couverture par polluant :
  no_moy               █████████░ 98%
  no2_moy              █████████░ 98%
  o3_moy               ████████░░ 87%
  pm10_moy             █████████░ 98%
  pm25_moy             █████████░ 92%
  so2_moy              ██░░░░░░░░ 29%
── Validation de dim_qualite_air ──
  ✅ Tous les codes dept sont valides (92 départements)
  ✅ Aucun doublon 

,dept,annee_mois,no_moy,no2_moy,o3_moy,pm10_moy,pm25_moy,so2_moy
0,01,2021-01,4.86,16.41,36.29,12.46,10.70,1.40
1,01,2021-02,4.81,14.39,45.65,21.63,12.85,0.98
2,01,2021-03,3.53,12.62,56.51,15.84,11.10,1.48
3,01,2021-04,1.34,6.84,74.79,12.92,8.30,0.78
4,01,2021-05,1.25,6.32,66.95,7.44,4.59,0.66
5,01,2021-06,0.91,6.51,72.05,13.48,8.08,1.28
6,01,2021-07,1.28,6.88,60.52,12.63,8.15,1.85
7,01,2021-08,1.27,5.98,55.78,9.93,5.99,1.32
8,01,2021-09,2.51,10.78,56.53,11.12,6.80,0.71
9,01,2021-10,5.56,11.99,44.56,11.55,8.39,0.47
